# DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation

## 1. Causal Inference

### 1.1 Potential Outcomes & Counterfactuals

For each individual $i$ with covariates $x_i$, treatment $w_i \in \{0,1\}$:

- $Y_i(1)$ — outcome if treated ($w_i = 1$)
- $Y_i(0)$ — outcome if not treated ($w_i = 0$)

Only one is observed: $y_i = w_i Y_i(1) + (1-w_i)Y_i(0)$. The unobserved outcome is the **counterfactual**.

### 1.2 Key Estimands

| Estimand | Definition | Meaning |
|---|---|---|
| **ITE / CATE** | $\tau(x) = \mathbb{E}[Y(1)-Y(0) \mid X=x]$ | Treatment effect for individuals with features $x$ |
| **ATE** | $\mathbb{E}[Y(1)-Y(0)]$ | Average effect over whole population |
| **ATT** | $\mathbb{E}[Y(1)-Y(0) \mid W=1]$ | Average effect on the treated |

**Treated Response**: $\mu_1(x) = \mathbb{E}[Y \mid W=1, X=x]$  
**Control Response**: $\mu_0(x) = \mathbb{E}[Y \mid W=0, X=x]$  
**Propensity Score**: $\pi(x) = P(W=1 \mid X=x)$

ITE is recovered as $\tau(x) = \mu_1(x) - \mu_0(x)$.

### 1.3 Three Assumptions

1. **Consistency**: $y_i = Y_i(w_i)$ — no interference between individuals
2. **Ignorability**: $Y(1),Y(0) \perp\!\!\!\perp W \mid X$ — no unmeasured confounders
3. **Overlap**: $0 < \pi(x) < 1$ for all $x$ — everyone could receive either treatment

### 1.4 Two Core Problems

| Problem | Cause | Example |
|---|---|---|
| **Treatment Bias** | Confounding → treated & control distributions differ | Inactive users get vouchers, active users don't |
| **Sample Imbalance** | $|T| \ll |C|$ or $|T| \gg |C|$ | Vouchers given to only 5% of users |

---

## 2. DESCN Architecture

DESCN addresses **both** treatment bias and sample imbalance through two integrated components.

### 2.1 Entire Space Network (ESN)

Instead of learning $\mu_1$ only on treated samples and $\mu_0$ only on control, ESN connects them via propensity:

$$\text{ESTR} = P(Y, W=1 \mid X) = \mu_1 \cdot \pi$$
$$\text{ESCR} = P(Y, W=0 \mid X) = \mu_0 \cdot (1-\pi)$$

ESTR and ESCR are trained on **all** samples. A treated sample contributes to learning $\mu_0$ (via ESCR), and vice versa.  
ESN implicitly performs Inverse Probability Weighting: $ATE = \mathbb{E}[\text{ESTR}/\pi] - \mathbb{E}[\text{ESCR}/(1-\pi)]$.

**ESN loss**: $\mathcal{L}_{ESN} = \alpha\mathcal{L}_{\pi} + \beta_1\mathcal{L}_{ESTR} + \beta_0\mathcal{L}_{ESCR}$

### 2.2 X-network

Introduces a **Pseudo Treatment Effect** $\tau'$ as a bridge between TR and CR, operating in logit space:

$$\mu_1' = \sigma\big(\sigma^{-1}(\mu_0) + \sigma^{-1}(\tau')\big) \quad \text{— Cross Treated Response}$$
$$\mu_0' = \sigma\big(\sigma^{-1}(\mu_1) - \sigma^{-1}(\tau')\big) \quad \text{— Cross Control Response}$$

Logit-space operations are numerically stable, keep outputs in $[0,1]$, and magnify uplift signals near boundaries.

**X-network losses**: $\mathcal{L}_{CrossTR}$ (on treated) and $\mathcal{L}_{CrossCR}$ (on control)

### 2.3 Model Variants

All variants share the same backbone — different loss weights produce different models:

| Model | $\alpha$ (prpsy) | $\beta_1,\beta_0$ (ESTR,ESCR) | $\gamma_1,\gamma_0$ (CrossTR,CrossCR) | $\lambda$ (IPM) | TR/CR |
|---|---|---|---|---|---|
| **TARNet** | 0 | 0, 0 | 0, 0 | 0 | 1, 1 |
| **CFR (MMD)** | 0 | 0, 0 | 0, 0 | 0.1 | 1, 1 |
| **X-network** | 0 | 0, 0 | 2, 1 | 0 | 2, 2 |
| **DESCN** | 0.5 | 0.5, 1 | 1, 0.5 | 0 | 0, 0 |

### 2.4 Full DESCN Loss

$$\mathcal{L}_{DESCN} = \alpha\mathcal{L}_{\pi} + \beta_1\mathcal{L}_{ESTR} + \beta_0\mathcal{L}_{ESCR} + \gamma_1\mathcal{L}_{CrossTR} + \gamma_0\mathcal{L}_{CrossCR}$$

> TR and CR are trained through ESTR/ESCR in the entire space, not directly ($h_1,h_0$ weights = 0).

---

## 3. Implementation

### 3.1 Setup

In [ ]:
import os, random

import numpy as np
import tensorflow as tf

# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

# ---- Device ----
gpu_devices = tf.config.list_physical_devices('GPU')
device_name = 'cuda' if gpu_devices else 'cpu'
print(f'TF {tf.__version__}  |  device: {device_name}')

### 3.2 Synthetic Data

This notebook uses a compact synthetic dataset to demonstrate the DESCN mechanics. It does not reproduce the full ACIC Epilepsy generator or Lazada production dataset from the paper. The implementation below keeps the same observed-data contract used by the original DESCN code: `x`, `t`, `yf`, `ycf`, `mu0`, `mu1`, `tau`, and `e`.

In [ ]:
def generate_data(sample_count=10000, feature_count=50, treatment_bias=True, imbalance=True):
    """
    Generate synthetic uplift data.
    
    Response functions are non-linear with heterogeneous treatment effects.
    Treatment assignment is biased by features when treatment_bias=True.
    """
    control_sample_count = sample_count // 2
    shifted_sample_count = sample_count - control_sample_count
    control_like_covariates = np.random.normal(-0.5, 1.0, (control_sample_count, feature_count))
    shifted_covariates = np.random.normal(0.5, 1.0, (shifted_sample_count, feature_count))
    covariates = np.concatenate([control_like_covariates, shifted_covariates], axis=0)
    
    control_response_logit = (
        0.3 * covariates[:, 0]
        + 0.2 * covariates[:, 1]
        - 0.1 * covariates[:, 2]
        + 0.05 * np.sum(covariates[:, 3:6], axis=1)
    )
    control_response_probability = 1 / (1 + np.exp(-control_response_logit))
    treatment_effect_logit_shift = 0.1 + 0.1 * np.tanh(covariates[:, 0])
    treated_response_logit = (
        np.log(control_response_probability / (1 - control_response_probability + 1e-9))
        + treatment_effect_logit_shift
    )
    treated_response_probability = 1 / (1 + np.exp(-treated_response_logit))
    individual_treatment_effect = treated_response_probability - control_response_probability
    
    if treatment_bias:
        treatment_propensity = np.clip(
            1 / (1 + np.exp(0.5 * covariates[:, 0] + 0.3 * covariates[:, 1])),
            0.02,
            0.95,
        )
    else:
        treatment_propensity = np.full(sample_count, 0.5)
    if imbalance:
        treatment_propensity = np.clip(treatment_propensity * 0.15, 0.02, 0.95)
    
    treatment = np.random.binomial(1, treatment_propensity).astype(np.float32)
    control_outcome = np.random.binomial(1, control_response_probability).astype(np.float32)
    treated_outcome = np.random.binomial(1, treated_response_probability).astype(np.float32)
    factual_outcome = treatment * treated_outcome + (1 - treatment) * control_outcome
    counterfactual_outcome = treatment * control_outcome + (1 - treatment) * treated_outcome
    
    randomized_flag = np.random.binomial(1, 0.2, sample_count).astype(np.float32)
    
    return {
        'x': covariates.astype(np.float32),
        't': treatment,
        'yf': factual_outcome,
        'ycf': counterfactual_outcome,
        'mu0': control_response_probability.astype(np.float32),
        'mu1': treated_response_probability.astype(np.float32),
        'tau': individual_treatment_effect.astype(np.float32),
        'e': randomized_flag,
    }

training_data = generate_data(20000, 50, treatment_bias=True, imbalance=True)
test_data = generate_data(5000, 50, treatment_bias=False, imbalance=False)

print(f'Train: {training_data["x"].shape[0]:,} samples, {training_data["x"].shape[1]} features')
print(f'  Treated: {training_data["t"].sum():.0f} ({training_data["t"].mean()*100:.1f}%)')
print(f'  Control: {(1-training_data["t"]).sum():.0f} ({(1-training_data["t"]).mean()*100:.1f}%)')
print(f'  Mean ITE: {training_data["tau"].mean():.4f}')
print(f'Test (RCT):  {test_data["x"].shape[0]:,} samples')
print(f'  Treated: {test_data["t"].sum():.0f} ({test_data["t"].mean()*100:.1f}%)')
print(f'  Mean ITE: {test_data["tau"].mean():.4f}')

### 3.3 Train / Validation / Test Split

In [ ]:
training_features = training_data['x']
training_outcomes = training_data['yf'].reshape(-1, 1)
training_treatments = training_data['t'].reshape(-1, 1)
training_randomized_flags = training_data['e'].reshape(-1, 1)

sample_count = len(training_features)
permuted_indices = np.random.permutation(sample_count)
validation_count = int(sample_count * 0.2)
validation_indices = permuted_indices[:validation_count]
train_indices = permuted_indices[validation_count:]

def to_tensor(array):
    return tf.constant(array, dtype=tf.float32)

features_train = to_tensor(training_features[train_indices])
outcomes_train = to_tensor(training_outcomes[train_indices])
treatments_train = to_tensor(training_treatments[train_indices])
randomized_flags_train = to_tensor(training_randomized_flags[train_indices])

features_validation = to_tensor(training_features[validation_indices])
outcomes_validation = to_tensor(training_outcomes[validation_indices])
treatments_validation = to_tensor(training_treatments[validation_indices])
randomized_flags_validation = to_tensor(training_randomized_flags[validation_indices])

features_test = to_tensor(test_data['x'])
outcomes_test = to_tensor(test_data['yf'].reshape(-1, 1))
treatments_test = to_tensor(test_data['t'].reshape(-1, 1))
randomized_flags_test = to_tensor(test_data['e'].reshape(-1, 1))
treatment_effect_test = to_tensor(test_data['tau'].reshape(-1, 1))

print(f'Train: {len(train_indices):,} | Val: {len(validation_indices):,} | Test: {len(features_test):,}')
print(
    f'Treated ratio - Train: {training_treatments[train_indices].mean():.3f} | '
    f'Val: {training_treatments[validation_indices].mean():.3f} | '
    f'Test: {test_data["t"].mean():.3f}'
)

### 3.4 Model Components

In [ ]:
from tensorflow.keras import layers

# ---- ShareNetwork: features -> shared representation h (L2-normalized) ----
class ShareNetwork(tf.keras.Model):
    def __init__(self, share_dim=128, base_dim=64, dropout=0.1, normalize='divide', **kwargs):
        super().__init__(**kwargs)
        self.normalize = normalize
        self.deep_network = tf.keras.Sequential([
            layers.BatchNormalization(),
            layers.Dense(share_dim, activation='elu'), layers.Dropout(dropout),
            layers.Dense(share_dim, activation='elu'), layers.Dropout(dropout),
            layers.Dense(base_dim, activation='elu'), layers.Dropout(dropout),
        ])

    def call(self, input_features, training=False):
        shared_representation = self.deep_network(input_features, training=training)
        if self.normalize == 'divide':
            representation_norm = tf.sqrt(
                tf.reduce_sum(tf.square(shared_representation), axis=1, keepdims=True) + 1e-9
            )
            shared_representation = shared_representation / representation_norm
        return shared_representation

# ---- Head: base_dim -> base_dim -> base_dim -> 1 (logit) ----
def make_head(dim=64, dropout=0.1, name='head'):
    return tf.keras.Sequential([
        layers.Dense(dim, activation='elu'), layers.Dropout(dropout),
        layers.Dense(dim, activation='elu'), layers.Dropout(dropout),
        layers.Dense(dim, activation='elu'), layers.Dropout(dropout),
        layers.Dense(1),
    ], name=name)

# ---- DESCN / ESX: shared backbone + 4 heads ----
class DESCN(tf.keras.Model):
    """
    Deep Entire Space Cross Networks.
    
    Forward returns 12 tensors:
      propensity_logit, entire_space_treated_response, entire_space_control_response,
      pseudo_effect_logit, treated_response_logit, control_response_logit,
      propensity, treated_response_probability, control_response_probability,
      treated_response_head, control_response_head, shared_representation
    
    The final ITE prediction follows the original code: treated_response_head - control_response_head.
    """
    def __init__(self, input_dim, share_dim=128, base_dim=64,
                 dropout=0.1, normalize='divide', **kwargs):
        super().__init__(**kwargs)
        self.share_network = ShareNetwork(share_dim, base_dim, dropout, normalize)
        self.propensity_head = make_head(base_dim, dropout, 'propensity')
        self.treated_response_head = make_head(base_dim, dropout, 'mu1')
        self.control_response_head = make_head(base_dim, dropout, 'mu0')
        self.pseudo_effect_head = make_head(base_dim, dropout, 'tau')
    
    def call(self, input_features, training=False):
        shared_representation = self.share_network(input_features, training=training)
        
        propensity_logit = self.propensity_head(shared_representation, training=training)
        treated_response_logit = self.treated_response_head(shared_representation, training=training)
        control_response_logit = self.control_response_head(shared_representation, training=training)
        pseudo_effect_logit = self.pseudo_effect_head(shared_representation, training=training)
        
        propensity = tf.clip_by_value(tf.nn.sigmoid(propensity_logit), 1e-3, 1 - 1e-3)
        treated_response_probability = tf.nn.sigmoid(treated_response_logit)
        control_response_probability = tf.nn.sigmoid(control_response_logit)
        
        entire_space_treated_response = propensity * treated_response_probability
        entire_space_control_response = (1.0 - propensity) * control_response_probability
        
        return (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            propensity,
            treated_response_probability,
            control_response_probability,
            treated_response_probability,
            control_response_probability,
            shared_representation,
        )

print('Model components defined.')

### 3.5 Loss Functions & IPM Distances

In [ ]:
# ---- BCE on probabilities ----
def binary_cross_entropy(labels, predictions):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    return -tf.reduce_mean(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))

# ---- Weighted BCE on probabilities ----
def weighted_binary_cross_entropy(labels, predictions, sample_weight):
    predictions = tf.clip_by_value(predictions, 1e-7, 1 - 1e-7)
    per_sample_loss = -(labels * tf.math.log(predictions) + (1 - labels) * tf.math.log(1 - predictions))
    return tf.reduce_mean(sample_weight * per_sample_loss)

# ---- BCE on logits with class weight (for propensity) ----
def weighted_logit_binary_cross_entropy(labels, logits, positive_weight):
    per_sample_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits)
    sample_weight = labels * positive_weight + (1.0 - labels)
    return tf.reduce_mean(sample_weight * per_sample_loss)

def pairwise_euclidean_distances(left_features, right_features):
    left_squared_norm = tf.reduce_sum(tf.square(left_features), axis=1, keepdims=True)
    right_squared_norm = tf.reduce_sum(tf.square(right_features), axis=1, keepdims=True)
    squared_distances = left_squared_norm - 2.0 * tf.matmul(left_features, right_features, transpose_b=True)
    squared_distances += tf.transpose(right_squared_norm)
    return tf.sqrt(tf.maximum(squared_distances, 1e-9))

def energy_distance_between_groups(shared_representations, treatments):
    treatment_vector = tf.reshape(treatments, (-1,))
    treated_indices = tf.where(treatment_vector > 0.5)[:, 0]
    control_indices = tf.where(treatment_vector < 0.5)[:, 0]

    def compute_distance():
        treated_representations = tf.gather(shared_representations, treated_indices)
        control_representations = tf.gather(shared_representations, control_indices)
        cross_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, control_representations))
        treated_distance = tf.reduce_mean(pairwise_euclidean_distances(treated_representations, treated_representations))
        control_distance = tf.reduce_mean(pairwise_euclidean_distances(control_representations, control_representations))
        return tf.maximum(2.0 * cross_distance - treated_distance - control_distance, 0.0)

    has_both_groups = tf.logical_and(tf.size(treated_indices) > 0, tf.size(control_indices) > 0)
    return tf.cond(has_both_groups, compute_distance, lambda: tf.constant(0.0, dtype=shared_representations.dtype))

# ---- IPM distances ----
def wasserstein_distance(shared_representations, treatments):
    return energy_distance_between_groups(shared_representations, treatments)

def mmd_distance(shared_representations, treatments):
    return energy_distance_between_groups(shared_representations, treatments)

def masked_binary_cross_entropy(labels, predictions, mask):
    return tf.cond(
        tf.reduce_any(mask),
        lambda: binary_cross_entropy(tf.boolean_mask(labels, mask), tf.boolean_mask(predictions, mask)),
        lambda: tf.constant(0.0, dtype=predictions.dtype),
    )

print('Loss functions defined.')

### 3.6 Evaluation

In [ ]:
from sklift.metrics import qini_auc_score

def evaluate(model, features, outcomes, treatments, randomized_flags, treatment_effect_true=None, weights=None):
    """Compute losses + metrics (AUUC, PEHE, ATE, ATT) on a dataset."""
    loss_weights = weights or {}
    propensity_weight = loss_weights.get('prpsy_w', 0)
    entire_treated_weight = loss_weights.get('escvr1_w', 0)
    entire_control_weight = loss_weights.get('escvr0_w', 0)
    treated_response_weight = loss_weights.get('h1_w', 0)
    control_response_weight = loss_weights.get('h0_w', 0)
    cross_treated_weight = loss_weights.get('mu1hat_w', 0)
    cross_control_weight = loss_weights.get('mu0hat_w', 0)
    imbalance_weight = loss_weights.get('imb_dist_w', 0)
    imbalance_type = loss_weights.get('imb_dist', 'wass')
    reweight_sample = loss_weights.get('reweight_sample', True)
    
    (
        propensity_logit,
        entire_space_treated_response,
        entire_space_control_response,
        pseudo_effect_logit,
        treated_response_logit,
        control_response_logit,
        propensity,
        treated_response_probability,
        control_response_probability,
        treated_response_head,
        control_response_head,
        shared_representation,
    ) = model(features, training=False)
    
    treatment_rate = tf.reduce_mean(treatments)
    if reweight_sample:
        sample_weight = treatments / (2 * treatment_rate) + (1 - treatments) / (2 * (1 - treatment_rate))
    else:
        sample_weight = tf.ones_like(treatments)
    
    non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags, tf.bool), tf.bool), (-1,))
    
    def mask_non_randomized(values):
        return tf.boolean_mask(values, non_randomized_mask)

    masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
    
    losses = {}
    if propensity_weight > 0:
        positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
        losses['propensity'] = propensity_weight * weighted_logit_binary_cross_entropy(
            mask_non_randomized(treatments),
            mask_non_randomized(propensity_logit),
            positive_weight,
        )
    if entire_treated_weight > 0:
        losses['estr'] = entire_treated_weight * weighted_binary_cross_entropy(
            mask_non_randomized(outcomes * treatments),
            mask_non_randomized(entire_space_treated_response),
            masked_sample_weight,
        )
    if entire_control_weight > 0:
        losses['escr'] = entire_control_weight * weighted_binary_cross_entropy(
            mask_non_randomized(outcomes * (1 - treatments)),
            mask_non_randomized(entire_space_control_response),
            masked_sample_weight,
        )
    
    treated_mask = treatments[:, 0] > 0.5
    control_mask = ~treated_mask
    if treated_response_weight > 0:
        losses['tr'] = treated_response_weight * masked_binary_cross_entropy(
            outcomes, treated_response_head, treated_mask
        )
    if control_response_weight > 0:
        losses['cr'] = control_response_weight * masked_binary_cross_entropy(
            outcomes, control_response_head, control_mask
        )
    
    if cross_treated_weight > 0:
        cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
        losses['cross_tr'] = cross_treated_weight * masked_binary_cross_entropy(
            outcomes, cross_treated_response, treated_mask
        )
    if cross_control_weight > 0:
        cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
        losses['cross_cr'] = cross_control_weight * masked_binary_cross_entropy(
            outcomes, cross_control_response, control_mask
        )
    
    if imbalance_weight > 0:
        imbalance_loss = wasserstein_distance(shared_representation, treatments)
        if imbalance_type == 'mmd':
            imbalance_loss = mmd_distance(shared_representation, treatments)
        losses['imb'] = imbalance_weight * imbalance_loss
    
    total_loss = tf.add_n(list(losses.values())) if losses else tf.constant(0.0)
    
    treatment_effect_predicted = (treated_response_head - control_response_head).numpy()
    outcomes_numpy = outcomes.numpy()
    treatments_numpy = treatments.numpy()
    
    results = {'total_loss': float(total_loss.numpy()), 'p_tau': treatment_effect_predicted}
    try:
        results['auuc'] = qini_auc_score(
            outcomes_numpy.reshape(-1),
            treatment_effect_predicted.reshape(-1),
            treatments_numpy.reshape(-1),
        )
    except Exception:
        results['auuc'] = 0.0
    
    flattened_treatments = treatments_numpy.flatten()
    if treatment_effect_true is not None:
        treatment_effect_true_numpy = treatment_effect_true.numpy()
        results['sqrt_pehe'] = np.sqrt(np.mean(np.square(treatment_effect_predicted - treatment_effect_true_numpy)))
        results['e_ate'] = np.abs(treatment_effect_predicted.mean() - treatment_effect_true_numpy.mean())
        if np.any(flattened_treatments == 1):
            results['e_att'] = np.abs(
                treatment_effect_predicted[flattened_treatments == 1].mean()
                - treatment_effect_true_numpy[flattened_treatments == 1].mean()
            )
    elif np.any(flattened_treatments == 1) and np.any(flattened_treatments == 0):
        flattened_outcomes = outcomes_numpy.flatten()
        observed_att = flattened_outcomes[flattened_treatments == 1].mean() - flattened_outcomes[flattened_treatments == 0].mean()
        results['e_att'] = np.abs(treatment_effect_predicted[flattened_treatments == 1].mean() - observed_att)
    
    for loss_name, loss_value in losses.items():
        results[f'loss_{loss_name}'] = float(loss_value.numpy())
    return results

print('evaluate() defined.')

### 3.7 Training

In [ ]:
@tf.function
def train_step(model, features_batch, treatments_batch, outcomes_batch, randomized_flags_batch, loss_weights, optimizer):
    with tf.GradientTape() as tape:
        (
            propensity_logit,
            entire_space_treated_response,
            entire_space_control_response,
            pseudo_effect_logit,
            treated_response_logit,
            control_response_logit,
            propensity,
            treated_response_probability,
            control_response_probability,
            treated_response_head,
            control_response_head,
            shared_representation,
        ) = model(features_batch, training=True)
        
        treatment_rate = tf.reduce_mean(treatments_batch)
        if loss_weights.get('reweight_sample', True):
            sample_weight = treatments_batch / (2 * treatment_rate) + (1 - treatments_batch) / (2 * (1 - treatment_rate))
        else:
            sample_weight = tf.ones_like(treatments_batch)
        
        non_randomized_mask = tf.reshape(tf.cast(~tf.cast(randomized_flags_batch, tf.bool), tf.bool), (-1,))

        def mask_non_randomized(values):
            return tf.boolean_mask(values, non_randomized_mask)

        masked_sample_weight = tf.reshape(tf.boolean_mask(tf.reshape(sample_weight, (-1,)), non_randomized_mask), (-1, 1))
        
        total_loss = tf.constant(0.0)
        if loss_weights['prpsy_w'] > 0:
            positive_weight = 1.0 / (2.0 * tf.maximum(treatment_rate, 1e-5))
            total_loss += loss_weights['prpsy_w'] * weighted_logit_binary_cross_entropy(
                mask_non_randomized(treatments_batch),
                mask_non_randomized(propensity_logit),
                positive_weight,
            )
        if loss_weights['escvr1_w'] > 0:
            total_loss += loss_weights['escvr1_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * treatments_batch),
                mask_non_randomized(entire_space_treated_response),
                masked_sample_weight,
            )
        if loss_weights['escvr0_w'] > 0:
            total_loss += loss_weights['escvr0_w'] * weighted_binary_cross_entropy(
                mask_non_randomized(outcomes_batch * (1 - treatments_batch)),
                mask_non_randomized(entire_space_control_response),
                masked_sample_weight,
            )
        
        treated_mask = treatments_batch[:, 0] > 0.5
        control_mask = ~treated_mask
        if loss_weights['h1_w'] > 0:
            total_loss += loss_weights['h1_w'] * masked_binary_cross_entropy(
                outcomes_batch, treated_response_head, treated_mask
            )
        if loss_weights['h0_w'] > 0:
            total_loss += loss_weights['h0_w'] * masked_binary_cross_entropy(
                outcomes_batch, control_response_head, control_mask
            )
        
        if loss_weights['mu1hat_w'] > 0:
            cross_treated_response = tf.nn.sigmoid(control_response_logit + pseudo_effect_logit)
            total_loss += loss_weights['mu1hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_treated_response, treated_mask
            )
        if loss_weights['mu0hat_w'] > 0:
            cross_control_response = tf.nn.sigmoid(treated_response_logit - pseudo_effect_logit)
            total_loss += loss_weights['mu0hat_w'] * masked_binary_cross_entropy(
                outcomes_batch, cross_control_response, control_mask
            )
        
        if loss_weights['imb_dist_w'] > 0:
            imbalance_loss = wasserstein_distance(shared_representation, treatments_batch)
            if loss_weights['imb_dist'] == 'mmd':
                imbalance_loss = mmd_distance(shared_representation, treatments_batch)
            total_loss += loss_weights['imb_dist_w'] * imbalance_loss
    
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return total_loss


def train(model, features_train, outcomes_train, treatments_train, randomized_flags_train,
          features_validation, outcomes_validation, treatments_validation, randomized_flags_validation,
          features_test, outcomes_test, treatments_test, randomized_flags_test, treatment_effect_test=None,
          weights=None, epochs=15, batch_size=500, lr=1e-3, l2=1e-2, name='Model'):
    """Train one model. Returns (history, final_test_results)."""
    loss_weights = dict(weights or {})
    loss_weights.setdefault('reweight_sample', True)
    loss_weights.setdefault('imb_dist', 'wass')
    for weight_name in ['prpsy_w','escvr1_w','escvr0_w','h1_w','h0_w','mu1hat_w','mu0hat_w','imb_dist_w']:
        loss_weights.setdefault(weight_name, 0.0)
    
    train_dataset = tf.data.Dataset.from_tensor_slices(
        (features_train, treatments_train, outcomes_train, randomized_flags_train)
    )
    train_dataset = train_dataset.shuffle(len(features_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    batch_count = len(features_train) // batch_size
    
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=tf.keras.optimizers.schedules.ExponentialDecay(lr, batch_count, 1.0),
        weight_decay=l2,
    )
    
    history = {'epoch': [], 'train_loss': [], 'val_loss': [],
               'auuc': [], 'sqrt_pehe': [], 'e_ate': []}
    
    print(f'[{name}] epochs={epochs}  batch={batch_size}  lr={lr}  l2={l2}')
    print(f'  weights: prpsy={loss_weights["prpsy_w"]} estr={loss_weights["escvr1_w"]} escr={loss_weights["escvr0_w"]} '
          f'h1={loss_weights["h1_w"]} h0={loss_weights["h0_w"]} '
          f'xTR={loss_weights["mu1hat_w"]} xCR={loss_weights["mu0hat_w"]} imb={loss_weights["imb_dist_w"]}')
    print(f'  train: {len(features_train):,}  val: {len(features_validation):,}  test: {len(features_test):,}  '
          f'treated(train): {tf.reduce_mean(treatments_train).numpy():.3f}')
    print('-' * 60)
    
    for epoch_index in range(epochs):
        epoch_losses = []
        for features_batch, treatments_batch, outcomes_batch, randomized_flags_batch in train_dataset:
            if tf.shape(features_batch)[0] < batch_size:
                continue
            batch_loss = train_step(
                model,
                features_batch,
                treatments_batch,
                outcomes_batch,
                randomized_flags_batch,
                loss_weights,
                optimizer,
            )
            epoch_losses.append(float(batch_loss.numpy()))
        
        average_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        validation_results = evaluate(
            model,
            features_validation,
            outcomes_validation,
            treatments_validation,
            randomized_flags_validation,
            weights=loss_weights,
        )
        test_results = evaluate(
            model,
            features_test,
            outcomes_test,
            treatments_test,
            randomized_flags_test,
            treatment_effect_true=treatment_effect_test,
            weights=loss_weights,
        )
        
        history['epoch'].append(epoch_index)
        history['train_loss'].append(average_loss)
        history['val_loss'].append(validation_results['total_loss'])
        history['auuc'].append(test_results.get('auuc', 0))
        history['sqrt_pehe'].append(test_results.get('sqrt_pehe', 0))
        history['e_ate'].append(test_results.get('e_ate', 0))
        
        if (epoch_index + 1) % max(1, epochs // 5) == 0 or epoch_index == 0:
            print(f'  epoch {epoch_index+1:3d}/{epochs}  '
                  f'train_loss={average_loss:.4f}  val_loss={validation_results["total_loss"]:.4f}  '
                  f'auuc={test_results.get("auuc",0):.4f}  sqrt_pehe={test_results.get("sqrt_pehe",0):.4f}  '
                  f'e_ate={test_results.get("e_ate",0):.4f}')
    
    print('-' * 60)
    final_test_results = evaluate(
        model,
        features_test,
        outcomes_test,
        treatments_test,
        randomized_flags_test,
        treatment_effect_true=treatment_effect_test,
        weights=loss_weights,
    )
    print(f'[{name}] Final: AUUC={final_test_results.get("auuc",0):.4f}  '
          f'sqrtPEHE={final_test_results.get("sqrt_pehe",0):.4f}  '
          f'e_ATE={final_test_results.get("e_ate",0):.4f}  e_ATT={final_test_results.get("e_att",0):.4f}')
    
    return history, final_test_results

print('train() defined.')

---

## 4. Model Training

Each model uses the same `DESCN` backbone. Only the loss weights differ (Section 2.3).

We train each model with `n_experiments=5` repeats and report mean ± standard error, following the paper's protocol.

### 4.1 TARNet [[Shalit et al., 2017]](https://arxiv.org/abs/1606.03976)

TARNet uses a shared representation with two heads (treated/control response), trained only on their respective sub-spaces. No entire space learning, no cross connections, no IPM regularization.

In [ ]:
TARNET_LOSS_WEIGHTS = {'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 1, 'h0_w': 1,
            'mu1hat_w': 0, 'mu0hat_w': 0,
            'imb_dist_w': 0}

RUN_COUNT, EPOCH_COUNT = 5, 15
tarnet_results = []

for run_index in range(RUN_COUNT):
    seed_everything(42 + run_index * 7)
    tarnet_model = DESCN(input_dim=training_data['x'].shape[1], share_dim=128, base_dim=64, dropout=0.1)
    _ = tarnet_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
    
    history, final_metrics = train(
        tarnet_model,
        features_train,
        outcomes_train,
        treatments_train,
        randomized_flags_train,
        features_validation,
        outcomes_validation,
        treatments_validation,
        randomized_flags_validation,
        features_test,
        outcomes_test,
        treatments_test,
        randomized_flags_test,
        treatment_effect_test,
        weights=TARNET_LOSS_WEIGHTS,
        epochs=EPOCH_COUNT,
        batch_size=500,
        lr=1e-3,
        l2=1e-2,
        name=f'TARNet run {run_index+1}/{RUN_COUNT}',
    )
    tarnet_results.append(final_metrics)

tarnet_sqrt_pehe_values = np.array([result['sqrt_pehe'] for result in tarnet_results])
tarnet_e_ate_values = np.array([result['e_ate'] for result in tarnet_results])
tarnet_auuc_values = np.array([result['auuc'] for result in tarnet_results])
print(f'TARNet ({RUN_COUNT} runs): sqrtPEHE={tarnet_sqrt_pehe_values.mean():.4f}+/-{tarnet_sqrt_pehe_values.std()/RUN_COUNT**0.5:.4f}  '
      f'e_ATE={tarnet_e_ate_values.mean():.4f}+/-{tarnet_e_ate_values.std()/RUN_COUNT**0.5:.4f}  '
      f'AUUC={tarnet_auuc_values.mean():.4f}+/-{tarnet_auuc_values.std()/RUN_COUNT**0.5:.4f}')

### 4.2 CFR (MMD) [[Shalit et al., 2017]](https://arxiv.org/abs/1606.03976)

CFR extends TARNet with an **IPM regularization** (MMD) on the shared representation to force treated and control distributions closer. This helps reduce treatment bias but does not address sample imbalance.

In [ ]:
CFR_MMD_LOSS_WEIGHTS = {'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
              'h1_w': 1, 'h0_w': 1,
              'mu1hat_w': 0, 'mu0hat_w': 0,
              'imb_dist_w': 0.1, 'imb_dist': 'mmd'}

cfr_mmd_results = []

for run_index in range(RUN_COUNT):
    seed_everything(42 + run_index * 7)
    cfr_mmd_model = DESCN(input_dim=training_data['x'].shape[1], share_dim=128, base_dim=64, dropout=0.1)
    _ = cfr_mmd_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
    
    history, final_metrics = train(
        cfr_mmd_model,
        features_train,
        outcomes_train,
        treatments_train,
        randomized_flags_train,
        features_validation,
        outcomes_validation,
        treatments_validation,
        randomized_flags_validation,
        features_test,
        outcomes_test,
        treatments_test,
        randomized_flags_test,
        treatment_effect_test,
        weights=CFR_MMD_LOSS_WEIGHTS,
        epochs=EPOCH_COUNT,
        batch_size=500,
        lr=1e-3,
        l2=1e-2,
        name=f'CFR(MMD) run {run_index+1}/{RUN_COUNT}',
    )
    cfr_mmd_results.append(final_metrics)

cfr_mmd_sqrt_pehe_values = np.array([result['sqrt_pehe'] for result in cfr_mmd_results])
cfr_mmd_e_ate_values = np.array([result['e_ate'] for result in cfr_mmd_results])
cfr_mmd_auuc_values = np.array([result['auuc'] for result in cfr_mmd_results])
print(f'CFR(MMD) ({RUN_COUNT} runs): sqrtPEHE={cfr_mmd_sqrt_pehe_values.mean():.4f}+/-{cfr_mmd_sqrt_pehe_values.std()/RUN_COUNT**0.5:.4f}  '
      f'e_ATE={cfr_mmd_e_ate_values.mean():.4f}+/-{cfr_mmd_e_ate_values.std()/RUN_COUNT**0.5:.4f}  '
      f'AUUC={cfr_mmd_auuc_values.mean():.4f}+/-{cfr_mmd_auuc_values.std()/RUN_COUNT**0.5:.4f}')

### 4.3 X-network

X-network adds the **Pseudo Treatment Effect** $\tau'$ and cross connections between TR and CR. This addresses sample imbalance by allowing both response functions to learn from each other through the PTE bridge. No entire space learning yet.

In [ ]:
XNETWORK_LOSS_WEIGHTS = {'prpsy_w': 0, 'escvr1_w': 0, 'escvr0_w': 0,
            'h1_w': 2, 'h0_w': 2,
            'mu1hat_w': 2, 'mu0hat_w': 1,
            'imb_dist_w': 0}

xnetwork_results = []

for run_index in range(RUN_COUNT):
    seed_everything(42 + run_index * 7)
    xnetwork_model = DESCN(input_dim=training_data['x'].shape[1], share_dim=128, base_dim=64, dropout=0.1)
    _ = xnetwork_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
    
    history, final_metrics = train(
        xnetwork_model,
        features_train,
        outcomes_train,
        treatments_train,
        randomized_flags_train,
        features_validation,
        outcomes_validation,
        treatments_validation,
        randomized_flags_validation,
        features_test,
        outcomes_test,
        treatments_test,
        randomized_flags_test,
        treatment_effect_test,
        weights=XNETWORK_LOSS_WEIGHTS,
        epochs=EPOCH_COUNT,
        batch_size=500,
        lr=1e-3,
        l2=1e-2,
        name=f'X-network run {run_index+1}/{RUN_COUNT}',
    )
    xnetwork_results.append(final_metrics)

xnetwork_sqrt_pehe_values = np.array([result['sqrt_pehe'] for result in xnetwork_results])
xnetwork_e_ate_values = np.array([result['e_ate'] for result in xnetwork_results])
xnetwork_auuc_values = np.array([result['auuc'] for result in xnetwork_results])
print(f'X-network ({RUN_COUNT} runs): sqrtPEHE={xnetwork_sqrt_pehe_values.mean():.4f}+/-{xnetwork_sqrt_pehe_values.std()/RUN_COUNT**0.5:.4f}  '
      f'e_ATE={xnetwork_e_ate_values.mean():.4f}+/-{xnetwork_e_ate_values.std()/RUN_COUNT**0.5:.4f}  '
      f'AUUC={xnetwork_auuc_values.mean():.4f}+/-{xnetwork_auuc_values.std()/RUN_COUNT**0.5:.4f}')

### 4.4 DESCN

DESCN combines **ESN** (entire space learning via $\text{ESTR} = \mu_1\pi$ and $\text{ESCR} = \mu_0(1-\pi)$) with **X-network** (cross connections via $\tau'$). TR and CR are trained only through ESTR/ESCR in the entire space ($h_1, h_0$ weights = 0). This addresses both treatment bias and sample imbalance simultaneously.

In [ ]:
DESCN_LOSS_WEIGHTS = {'prpsy_w': 0.5, 'escvr1_w': 0.5, 'escvr0_w': 1.0,
             'h1_w': 0, 'h0_w': 0,
             'mu1hat_w': 1.0, 'mu0hat_w': 0.5,
             'imb_dist_w': 0}

descn_results = []

for run_index in range(RUN_COUNT):
    seed_everything(42 + run_index * 7)
    descn_model = DESCN(input_dim=training_data['x'].shape[1], share_dim=128, base_dim=64, dropout=0.1)
    _ = descn_model(tf.zeros((2, training_data['x'].shape[1])), training=False)
    
    history, final_metrics = train(
        descn_model,
        features_train,
        outcomes_train,
        treatments_train,
        randomized_flags_train,
        features_validation,
        outcomes_validation,
        treatments_validation,
        randomized_flags_validation,
        features_test,
        outcomes_test,
        treatments_test,
        randomized_flags_test,
        treatment_effect_test,
        weights=DESCN_LOSS_WEIGHTS,
        epochs=EPOCH_COUNT,
        batch_size=500,
        lr=1e-3,
        l2=1e-2,
        name=f'DESCN run {run_index+1}/{RUN_COUNT}',
    )
    descn_results.append(final_metrics)

descn_sqrt_pehe_values = np.array([result['sqrt_pehe'] for result in descn_results])
descn_e_ate_values = np.array([result['e_ate'] for result in descn_results])
descn_auuc_values = np.array([result['auuc'] for result in descn_results])
print(f'DESCN ({RUN_COUNT} runs): sqrtPEHE={descn_sqrt_pehe_values.mean():.4f}+/-{descn_sqrt_pehe_values.std()/RUN_COUNT**0.5:.4f}  '
      f'e_ATE={descn_e_ate_values.mean():.4f}+/-{descn_e_ate_values.std()/RUN_COUNT**0.5:.4f}  '
      f'AUUC={descn_auuc_values.mean():.4f}+/-{descn_auuc_values.std()/RUN_COUNT**0.5:.4f}')

---

## 5. Results Comparison

Reproducing **Table 2** from the paper: model performance on synthetic data with ground truth ITE.

In [ ]:
import matplotlib.pyplot as plt

baseline_model_name = 'CFR(MMD)'
model_names = ['TARNet', baseline_model_name, 'X-network', 'DESCN']
sqrt_pehe_values_by_model = [
    tarnet_sqrt_pehe_values,
    cfr_mmd_sqrt_pehe_values,
    xnetwork_sqrt_pehe_values,
    descn_sqrt_pehe_values,
]
e_ate_values_by_model = [
    tarnet_e_ate_values,
    cfr_mmd_e_ate_values,
    xnetwork_e_ate_values,
    descn_e_ate_values,
]
auuc_values_by_model = [
    tarnet_auuc_values,
    cfr_mmd_auuc_values,
    xnetwork_auuc_values,
    descn_auuc_values,
]

baseline_sqrt_pehe = cfr_mmd_sqrt_pehe_values.mean()
baseline_auuc = cfr_mmd_auuc_values.mean()

print('=' * 100)
print(f'MODEL COMPARISON ({RUN_COUNT} runs, mean +/- std error)')
print('=' * 100)
print(f'{"Model":<16s} {"sqrtPEHE":<22s} {"Impr%":>8s}  {"e_ATE":<22s} {"AUUC":<22s} {"Impr%":>8s}  {"e_ATT":<22s}')
print('-' * 105)

results_by_model = [tarnet_results, cfr_mmd_results, xnetwork_results, descn_results]
for model_name, sqrt_pehe_values, e_ate_values, auuc_values, model_results in zip(
    model_names,
    sqrt_pehe_values_by_model,
    e_ate_values_by_model,
    auuc_values_by_model,
    results_by_model,
):
    e_att_values = np.array([result.get('e_att', 0) for result in model_results])
    sqrt_pehe_mean = sqrt_pehe_values.mean()
    sqrt_pehe_standard_error = sqrt_pehe_values.std() / RUN_COUNT**0.5
    e_ate_mean = e_ate_values.mean()
    e_ate_standard_error = e_ate_values.std() / RUN_COUNT**0.5
    auuc_mean = auuc_values.mean()
    auuc_standard_error = auuc_values.std() / RUN_COUNT**0.5
    e_att_mean = e_att_values.mean()
    e_att_standard_error = e_att_values.std() / RUN_COUNT**0.5
    
    sqrt_pehe_improvement = (sqrt_pehe_mean - baseline_sqrt_pehe) / baseline_sqrt_pehe * 100
    auuc_improvement = (auuc_mean - baseline_auuc) / baseline_auuc * 100
    best_marker = '  <<' if model_name == 'DESCN' else ''
    print(f'{model_name:<16s} {sqrt_pehe_mean:.4f} +/- {sqrt_pehe_standard_error:.4f}     {sqrt_pehe_improvement:>+6.1f}%  '
          f'{e_ate_mean:.4f} +/- {e_ate_standard_error:.4f}     {auuc_mean:.4f} +/- {auuc_standard_error:.4f}     {auuc_improvement:>+6.1f}%  '
          f'{e_att_mean:.4f} +/- {e_att_standard_error:.4f}{best_marker}')

print()
print(f'Improvement over {baseline_model_name} baseline.')
print('sqrtPEHE/e_ATE/e_ATT: negative improvement = lower error (better).')
print('AUUC: positive improvement = higher uplift ranking (better).')
print()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bar_colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']

for axis_index, (metric_name, metric_values_by_model, axis_label) in enumerate([
    ('sqrt_pehe', sqrt_pehe_values_by_model, 'sqrtPEHE'),
    ('e_ate', e_ate_values_by_model, 'epsilon_ATE'),
    ('auuc', auuc_values_by_model, 'AUUC'),
]):
    metric_means = [model_values.mean() for model_values in metric_values_by_model]
    metric_errors = [model_values.std() / RUN_COUNT**0.5 for model_values in metric_values_by_model]
    axes[axis_index].bar(model_names, metric_means, yerr=metric_errors, color=bar_colors, capsize=5)
    axes[axis_index].set_title(axis_label)
    axes[axis_index].grid(axis='y', alpha=0.3)
    for model_index, metric_value in enumerate(metric_means):
        axes[axis_index].text(
            model_index,
            metric_value + metric_errors[model_index],
            f'{metric_value:.4f}',
            ha='center',
            va='bottom',
            fontsize=8,
        )

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to model_comparison.png')

---

## 6. Summary

### What DESCN achieves

1. **ESN** learns response functions in the **entire sample space** via $\text{ESTR} = \mu_1\pi$ and $\text{ESCR} = \mu_0(1-\pi)$. A treated sample teaches control outcomes, and vice versa. Implicitly performs Inverse Probability Weighting.

2. **X-network** bridges treated and control responses through a **Pseudo Treatment Effect** $\tau'$, operating in logit space for numerical stability. Cross predictions create counterfactual estimates that balance learning when one group is much smaller than the other.

3. **Unified architecture**: TARNet, CFR, X-network, and DESCN all use the same backbone. Only loss weights differ, making the framework a flexible testbed for uplift modeling.

### References

- **Zhong et al. (2022)**. *DESCN: Deep Entire Space Cross Networks for Individual Treatment Effect Estimation*. KDD '22. [arXiv:2207.09920](https://arxiv.org/abs/2207.09920)
- **Shalit et al. (2017)**. *Estimating individual treatment effect: generalization bounds and algorithms*. ICML '17. (TARNet/CFR)
- **Kunzel et al. (2019)**. *Metalearners for estimating heterogeneous treatment effects using machine learning*. PNAS. (X-learner)
- **Rosenbaum & Rubin (1983)**. *The central role of the propensity score in observational studies for causal effects*. Biometrika. (Propensity Score / IPW)
- **Original DESCN code**: [github.com/kailiang-zhong/DESCN](https://github.com/kailiang-zhong/DESCN)